In [1]:

import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim

train_df = pd.read_csv('../data/final_training_data.csv')
print(train_df.columns)
features = train_df[["YearStart", "LocationAbbr", "Stratification1"]]
target = train_df['DataValue']
print(train_df.head())

Index(['YearStart', 'LocationAbbr', 'DataValue', 'Stratification1'], dtype='object')
   YearStart LocationAbbr  DataValue  \
0       2019           AZ       97.2   
1       2019           AK      132.3   
2       2019           AR      230.9   
3       2019           AZ      140.3   
4       2019           AZ       65.4   

                                  Stratification1  
0  American Indian or Alaska Native, non-Hispanic  
1                             Black, non-Hispanic  
2                             Black, non-Hispanic  
3                             Black, non-Hispanic  
4                             Asian, non-Hispanic  


In [2]:
categorical_preprocessing = Pipeline([
    ('ohe', OneHotEncoder())
])
numeric_preprocessing = Pipeline([
    ('scaler', StandardScaler())  # Scale numeric features to improve performance
])

# Combine preprocessing steps into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_preprocessing, ['LocationAbbr', 'Stratification1']),
        ('num', numeric_preprocessing, ['YearStart'])
    ])

# Apply transformations to the feature matrix
features_transformed = preprocessor.fit_transform(features)

X_train, X_test, y_train, y_test = train_test_split(features_transformed, target, test_size=0.2, random_state=42)



In [3]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
predictions = linear_model.predict(X_test)

mse = mean_squared_error(y_test, predictions)
print(f"Mean Squared Error: {mse}")
r_squared = linear_model.score(X_test, y_test)
print(f"R-squared: {r_squared}")


Mean Squared Error: 1095.6369955937344
R-squared: 0.8868755145768086


In [4]:

forest_model = RandomForestRegressor(n_estimators=100, random_state=42)
forest_model.fit(X_train, y_train)
predictions = forest_model.predict(X_test)

mse = mean_squared_error(y_test, predictions)
print(f"Mean Squared Error: {mse}")
r_squared = forest_model.score(X_test, y_test)
print(f"R-squared: {r_squared}")

Mean Squared Error: 272.3409810402683
R-squared: 0.9718808022513586


In [5]:
features = train_df[["YearStart", "LocationAbbr", "Stratification1"]]
target = train_df['DataValue'].values

categorical_preprocessing = OneHotEncoder()
numeric_preprocessing = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_preprocessing, ['LocationAbbr', 'Stratification1']),
        ('num', numeric_preprocessing, ['YearStart'])
    ])

features_transformed = preprocessor.fit_transform(features).toarray()
features_transformed = torch.tensor(features_transformed, dtype=torch.float32)
target = torch.tensor(target, dtype=torch.float32).view(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(features_transformed.numpy(), target.numpy(), test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(features_transformed.shape[1], 50)
        self.dropout1 = nn.Dropout(0.25)  # 25% dropout rate
        self.fc2 = nn.Linear(50, 20)
        self.dropout2 = nn.Dropout(0.25)  # Another 25% dropout rate
        self.fc3 = nn.Linear(20, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

neural_network = Net()


criterion = nn.MSELoss()
optimizer = optim.Adam(neural_network.parameters(), lr=0.001)

def train_model(model, train_loader, criterion, optimizer, num_epochs=50):
    neural_network.train()
    for epoch in range(num_epochs):
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = neural_network(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

train_model(neural_network, train_loader, criterion, optimizer)

def r2_score(y_true, y_pred):
    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - torch.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot
    return r2.item()

def evaluate_model(neural_network, test_loader, criterion):
    neural_network.eval()
    total_loss = 0
    total_r2 = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = neural_network(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            total_r2 += r2_score(labels, outputs) * inputs.size(0)
        avg_loss = total_loss / len(test_loader.dataset)
        avg_r2 = total_r2 / len(test_loader.dataset)
    print(f'Test MSE: {avg_loss}')
    print(f'Test R^2: {avg_r2}')

# Evaluate the model on the test data
evaluate_model(neural_network, test_loader, criterion)

Epoch 1/50, Loss: 17169.744140625
Epoch 2/50, Loss: 27941.3125
Epoch 3/50, Loss: 36972.05859375
Epoch 4/50, Loss: 19000.181640625
Epoch 5/50, Loss: 30528.677734375
Epoch 6/50, Loss: 24883.609375
Epoch 7/50, Loss: 23056.662109375
Epoch 8/50, Loss: 29658.765625
Epoch 9/50, Loss: 15546.4853515625
Epoch 10/50, Loss: 27336.224609375
Epoch 11/50, Loss: 20663.216796875
Epoch 12/50, Loss: 17997.33984375
Epoch 13/50, Loss: 27769.630859375
Epoch 14/50, Loss: 25469.580078125
Epoch 15/50, Loss: 16942.01953125
Epoch 16/50, Loss: 11099.6826171875
Epoch 17/50, Loss: 33895.51953125
Epoch 18/50, Loss: 6940.97216796875
Epoch 19/50, Loss: 12409.5869140625
Epoch 20/50, Loss: 7828.51904296875
Epoch 21/50, Loss: 6846.91455078125
Epoch 22/50, Loss: 13491.4375
Epoch 23/50, Loss: 9200.71875
Epoch 24/50, Loss: 7828.634765625
Epoch 25/50, Loss: 11062.728515625
Epoch 26/50, Loss: 6713.77734375
Epoch 27/50, Loss: 4878.755859375
Epoch 28/50, Loss: 4243.21533203125
Epoch 29/50, Loss: 5542.55859375
Epoch 30/50, Loss:

In [7]:
from joblib import dump, load
from torch import save, load as torch_load

dump(linear_model, 'linear_regression_model.joblib')
dump(forest_model, 'random_forest_model.joblib')
save(neural_network.state_dict(), 'neural_network_model.pth')